In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, balanced_accuracy_score
import pandas as pd
import numpy as np
import math
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
import os
import pickle
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.metrics import confusion_matrix,recall_score,matthews_corrcoef,roc_curve,roc_auc_score,auc,precision_recall_curve
from sklearn.metrics import accuracy_score,roc_curve
from sklearn.metrics import cohen_kappa_score, accuracy_score, roc_auc_score, precision_score, recall_score, balanced_accuracy_score
from sklearn import metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:

import torch.nn as nn

class DynamicConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_list=[3,5,7]):
        super().__init__()
        self.kernel_list = kernel_list
        self.out_channels = out_channels
        self.in_channels = in_channels
        
        
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels, kernel_size=k, padding=k//2)
            for k in kernel_list
        ])
        
       
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(in_channels, len(kernel_list)),
            nn.Softmax(dim=1)
        )
        
    def forward(self, x):
        )
        if x.dim() == 2:  # (batch, seq_len)
            x = x.unsqueeze(1)  # -> (batch, 1, seq_len)
        elif x.dim() == 3 and x.size(1) != self.in_channels:  # (batch, seq_len, channels)
            x = x.permute(0, 2, 1)  # -> (batch, channels, seq_len)
        
        
        attn_weights = self.attention(x).unsqueeze(-1).unsqueeze(-1)  # [B, num_kernels, 1, 1]
        
       
        features = []
        for conv in self.conv_layers:
            features.append(conv(x).unsqueeze(1))  # [B, 1, C, L]
        
        features = torch.cat(features, dim=1)  # [B, num_kernels, C, L]
        output = torch.sum(features * attn_weights, dim=1)  # [B, C, L]
        
        return output
    


class CNNLayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CNNLayer, self).__init__()
        
        
        self.dynamic_conv = DynamicConv1d(1, 16, kernel_list=[3,5,7])
      
        self.conv_fuse = nn.Conv1d(16, 64, kernel_size=1)
        
      
        self.shortcut = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=1),
            nn.BatchNorm1d(64)
        )
        
        
        self.conv2 = nn.Conv1d(64, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(32, 1, kernel_size=1)
      
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.adaptive_pool = nn.AdaptiveAvgPool1d(hidden_size)
        
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        
    def forward(self, x):

       
        identity = x if x.dim() == 3 and x.size(1) == 1 else x.unsqueeze(1)
        
     
        x = self.relu(self.dynamic_conv(x))  
       
        x = self.relu(self.conv_fuse(x))    
        
   
        shortcut = self.shortcut(identity)
        x = x + shortcut
        x = self.relu(x)
        
   
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.conv3(x)
        x = self.adaptive_pool(x)
        
        return x
class LayerNorm(nn.Module):
    def __init__(self, hidden_size, variance_epsilon=1e-12):

        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_size))
        self.beta = nn.Parameter(torch.zeros(hidden_size))
        self.variance_epsilon = variance_epsilon

    def forward(self, x):
        u = x.mean(-1, keepdim=True)
        # Normalize input_tensor
        s = (x - u).pow(2).mean(-1, keepdim=True)
        # Apply scaling and bias
        x = (x - u) / torch.sqrt(s + self.variance_epsilon)
        return self.gamma * x + self.beta


class SelfAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_attention_heads, attention_probs_dropout_prob):
        super(SelfAttention, self).__init__()
        self.input_size = input_size   
        self.hidden_size = hidden_size  
        self.num_attention_heads = num_attention_heads
        self.attention_head_size = int(hidden_size / num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        
        self.query = nn.Linear(1, self.all_head_size)
        self.key = nn.Linear(1, self.all_head_size)
        self.value = nn.Linear(1, self.all_head_size)
        self.output_projection = nn.Linear(self.all_head_size, 1)  
        self.dropout = nn.Dropout(attention_probs_dropout_prob)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states, attention_mask=None):
        #print(f"Input shape: {hidden_states.shape}") 
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)
        
        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)
        
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        attention_probs_0 = nn.Softmax(dim=-1)(attention_scores)
        attention_probs = self.dropout(attention_probs_0)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        
  
        # print(f"context_layer.shape: {context_layer.shape}")
        context_layer = self.output_projection(context_layer)
        # context_layer = context_layer.permute(0, 2, 1)
        # print(context_layer.shape)
        return context_layer , attention_probs_0


class CrossAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_attention_heads, attention_probs_dropout_prob):
        super(CrossAttention, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_attention_heads = num_attention_heads
        self.attention_head_size = int(hidden_size / num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        # Change the input dimension of the linear layers to 1
        self.query = nn.Linear(1, self.all_head_size)
        self.key = nn.Linear(1, self.all_head_size)
        self.value = nn.Linear(1, self.all_head_size)
        
        # The output projection will also output a feature dimension of 1
        self.output_projection = nn.Linear(self.all_head_size, 1) 
        self.dropout = nn.Dropout(attention_probs_dropout_prob)


    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, query_input, key_value_input, attention_mask=None):
        mixed_query_layer = self.query(query_input)
        mixed_key_layer = self.key(key_value_input)
        mixed_value_layer = self.value(key_value_input)
        
        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)
        
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size) 
        
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        attention_probs_0 = nn.Softmax(dim=-1)(attention_scores) 
        attention_probs = self.dropout(attention_probs_0)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        
      
        context_layer = self.output_projection(context_layer)
        
        # context_layer = context_layer.permute(0, 2, 1) 
        # print(context_layer.shape)
        return context_layer, attention_probs_0


class SelfOutput(nn.Module):#Add/Norm
    def __init__(self, hidden_size, hidden_dropout_prob):
        super(SelfOutput, self).__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.LayerNorm = LayerNorm(hidden_size)
        self.dropout = nn.Dropout(hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        # print("Input shape:", hidden_states.shape)
        hidden_states = self.dense(hidden_states)
        # print("After dense:", hidden_states.shape)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states)
        # print("input_tensor shape:", input_tensor.shape) 
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        # print(hidden_states.shape)
        return hidden_states    
    
class CrossOutput(nn.Module):
    def __init__(self, hidden_size, hidden_dropout_prob):
        super(CrossOutput, self).__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.LayerNorm = LayerNorm(hidden_size)
        self.dropout = nn.Dropout(hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


class Output(nn.Module): 
    def __init__(self, intermediate_size, hidden_size, hidden_dropout_prob):
        super(Output, self).__init__()
        self.dense = nn.Linear(intermediate_size, hidden_size)
        self.LayerNorm = LayerNorm(hidden_size)
        self.dropout = nn.Dropout(hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states
    



class Attention_D(nn.Module):
    def __init__(self, input_size, hidden_size, num_attention_heads, attention_probs_dropout_prob, hidden_dropout_prob):
        super(Attention_D, self).__init__()
        self.cnn = CNNLayer(input_size, hidden_size)
     
        self.self = SelfAttention(hidden_size, hidden_size, num_attention_heads, attention_probs_dropout_prob)
        self.output = SelfOutput(hidden_size, hidden_dropout_prob)

    def forward(self, input_tensor, attention_mask):
        cnn_output_1 = self.cnn(input_tensor)  
        #print(cnn_output_1.shape)
        cnn_output = cnn_output_1.permute(0, 2, 1)  # (bs, seq_len, hidden_size)
        #print(cnn_output.shape)
        self_output, attention_probs_0 = self.self(cnn_output, attention_mask)
        self_output = self_output.permute(0, 2, 1)
        attention_output = self.output(self_output, cnn_output_1)
        return attention_output, attention_probs_0 

    

class Attention_CD(nn.Module):
    def __init__(self, input_size, hidden_size, num_attention_heads, attention_probs_dropout_prob, hidden_dropout_prob):
        super(Attention_CD, self).__init__()
        self.self = CrossAttention(input_size, hidden_size, num_attention_heads, attention_probs_dropout_prob)
        self.output = CrossOutput(hidden_size, hidden_dropout_prob)

    def forward(self, drug, cell, drug_attention_mask, cell_attention_mask):
        # Transpose the inputs before passing to CrossAttention
        drug_transposed = drug.permute(0, 2, 1)
        cell_transposed = cell.permute(0, 2, 1)

        drug_self_output, drug_attention_probs_0 = self.self(drug_transposed, cell_transposed, drug_attention_mask)
        cell_self_output, cell_attention_probs_0 = self.self(cell_transposed, drug_transposed, cell_attention_mask)

        # Transpose the outputs back to the original shape
        drug_self_output = drug_self_output.permute(0, 2, 1)
        cell_self_output = cell_self_output.permute(0, 2, 1)

        drug_attention_output = self.output(drug_self_output, drug)
        cell_attention_output = self.output(cell_self_output, cell)
        return drug_attention_output, cell_attention_output, drug_attention_probs_0, cell_attention_probs_0 
    
class Intermediate(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super(Intermediate, self).__init__()
        self.dense = nn.Linear(hidden_size, intermediate_size)

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = F.relu(hidden_states)
        return hidden_states

 
    
class SelfEncoder_D(nn.Module):
    def __init__(self, input_size, hidden_size, intermediate_size, num_attention_heads, attention_probs_dropout_prob, hidden_dropout_prob):
        super(SelfEncoder_D, self).__init__()
        self.attention = Attention_D(input_size, hidden_size, num_attention_heads,
                                   attention_probs_dropout_prob, hidden_dropout_prob)
        self.intermediate = Intermediate(hidden_size, intermediate_size)
        self.output = Output(intermediate_size, hidden_size, hidden_dropout_prob)

    def forward(self, hidden_states, attention_mask):
        attention_output, attention_probs_0 = self.attention(hidden_states, attention_mask)#[32,1,4937]->[32,1,256]
        # print(attention_output.shape)
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output, attention_probs_0
 
    
class EncoderCD(nn.Module):
    def __init__(self, hidden_size, intermediate_size, num_attention_heads, attention_probs_dropout_prob, hidden_dropout_prob):
        super(EncoderCD, self).__init__()
        self.attention_DC = Attention_CD(hidden_size,hidden_size, num_attention_heads,
                                   attention_probs_dropout_prob, hidden_dropout_prob)
        self.intermediate = Intermediate(hidden_size, intermediate_size)
        self.output = Output(intermediate_size, hidden_size, hidden_dropout_prob)

    def forward(self, drug, cell, drug_attention_mask, cell_attention_mask):
        drug_attention_output, cell_attention_output, drug_attention_probs_0, cell_attention_probs_0  = self.attention_DC(drug, cell, drug_attention_mask, cell_attention_mask)
        drug_intermediate_output = self.intermediate(drug_attention_output)
        drug_layer_output = self.output(drug_intermediate_output, drug_attention_output)
        cell_intermediate_output = self.intermediate(cell_attention_output)
        cell_layer_output = self.output(cell_intermediate_output, cell_attention_output)
        return drug_layer_output, cell_layer_output, drug_attention_probs_0, cell_attention_probs_0 



class PreNN(nn.Module):
    def __init__(self, input_size):
        super(PreNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 1024)
        self.fc2 = nn.Linear(1024, 256)
        self.fc3 = nn.Linear(256, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        #x = self.sigmoid(self.fc3(x))
        return x


In [ ]:
import torch


def drug_feat(tensor, device):
    tensor = tensor.float().to(device)
    if tensor.dim() == 2:
        tensor = tensor.unsqueeze(1)
    mask = torch.ones(tensor.size(0), 1).to(device)
    expanded_mask = (1.0 - mask.unsqueeze(1).unsqueeze(2)) * -10000.0
    return tensor, expanded_mask

# class GraphCrossSynergy_NoSelfAtt(nn.Module):
#     def __init__(self, input_size_drug=384, input_size_cell=977, hidden_self_size=256, hidden_cross_size=256, num_heads=4, dropout=0.3):
#         super(GraphCrossSynergy_NoSelfAtt, self).__init__()


#     
#         self.drug_a_proj = nn.Linear(input_size_drug, hidden_self_size)
#         self.drug_b_proj = nn.Linear(input_size_drug, hidden_self_size)
#         self.cell_proj = nn.Linear(input_size_cell, hidden_self_size)
#     
#       
#         self.cross_attention_drug_a_cell = Attention_CD(hidden_self_size, hidden_cross_size, num_heads, dropout, dropout)
#         self.cross_attention_drug_b_cell = Attention_CD(hidden_self_size, hidden_cross_size, num_heads, dropout, dropout)
        
#  
#         self.pre = PreNN(hidden_cross_size * 4)

#     def forward(self, drug_a, drug_b, cell):
#       
#         drug_a = drug_a.float()
#         drug_b = drug_b.float()
#         cell = cell.float()
#         drug_a = drug_a.unsqueeze(1)
#         drug_b = drug_b.unsqueeze(1)
#         cell = cell.unsqueeze(1)

#         drug_a, drug_a_mask = drug_feat(drug_a, device)
#         drug_b, drug_b_mask = drug_feat(drug_b, device)
#         cell, cell_mask = drug_feat(cell, device)
 
#  
#         # drug_a_features, _ = self.self_attention_drug_a(drug_a, drug_a_mask)
#         # drug_b_features, _ = self.self_attention_drug_b(drug_b, drug_b_mask)
#         # cell_features, _ = self.self_attention_cell(cell, cell_mask)

#       
#         drug_a_features = self.drug_a_proj(drug_a.squeeze(1)).unsqueeze(1)
#         drug_b_features = self.drug_b_proj(drug_b.squeeze(1)).unsqueeze(1)
#         cell_features = self.cell_proj(cell.squeeze(1)).unsqueeze(1)
#        
        
#        
#         drug_a_cell_features, cellA_attention_output, _, _ = self.cross_attention_drug_a_cell(drug_a_features, cell_features, drug_a_mask, cell_mask)
#         drug_b_cell_features, cellB_attention_output, _, _ = self.cross_attention_drug_b_cell(drug_b_features, cell_features, drug_b_mask, cell_mask)

#         combined_features = torch.cat([drug_a_cell_features, drug_b_cell_features, cellA_attention_output, cellB_attention_output], dim=-1)
        
#       
#         output = self.pre(combined_features)
#         return output

class GraphCrossSynergy_NoCrossAtt(nn.Module):
    def __init__(self, input_size_drug=384, input_size_cell=977, hidden_self_size=256, hidden_cross_size=256,num_heads=4, dropout=0.3):
        super(GraphCrossSynergy_NoCrossAtt, self).__init__()

        intermediate_size= hidden_self_size * 2
       
        self.self_attention_drug_a = SelfEncoder_D(input_size_drug, hidden_self_size, intermediate_size, num_heads, dropout, dropout)
        self.self_attention_drug_b = SelfEncoder_D(input_size_drug, hidden_self_size, intermediate_size, num_heads, dropout, dropout)
        self.self_attention_cell = SelfEncoder_D(input_size_cell, hidden_self_size, intermediate_size, num_heads, dropout, dropout)
        
       
        # self.cross_attention_drug_a_cell = Attention_CD(...)
        # self.cross_attention_drug_b_cell = Attention_CD(...)
       
        self.pre = PreNN(hidden_self_size * 3)
    

    def forward(self, drug_a, drug_b, cell):
      
        drug_a = drug_a.float()
        drug_b = drug_b.float()
        cell = cell.float()
        drug_a = drug_a.unsqueeze(1)
        drug_b = drug_b.unsqueeze(1)
        cell = cell.unsqueeze(1)

        drug_a, drug_a_mask = drug_feat(drug_a, device)
        drug_b, drug_b_mask = drug_feat(drug_b, device)
        cell, cell_mask = drug_feat(cell, device)
 
     
        drug_a_features, _ = self.self_attention_drug_a(drug_a, drug_a_mask)
        drug_b_features, _ = self.self_attention_drug_b(drug_b, drug_b_mask)
        cell_features, _ = self.self_attention_cell(cell, cell_mask)
        
       
        # drug_a_cell_features, cellA_attention_output, _, _ = self.cross_attention_drug_a_cell(...)
        # drug_b_cell_features, cellB_attention_output, _, _ = self.cross_attention_drug_b_cell(...)

        
        combined_features = torch.cat([
            drug_a_features.squeeze(1), 
            drug_b_features.squeeze(1), 
            cell_features.squeeze(1)
        ], dim=-1)
        
       
        output = self.pre(combined_features)
        return output

class GraphCrossSynergy(nn.Module):
    def __init__(self, input_size_drug=384, input_size_cell=977, hidden_self_size=256, hidden_cross_size=256,num_heads=4, dropout=0.3):
        super(GraphCrossSynergy, self).__init__()

        intermediate_size= hidden_self_size * 3 #2→3
      
        self.self_attention_drug_a = SelfEncoder_D(input_size_drug, hidden_self_size, intermediate_size, num_heads, dropout, dropout)
        self.self_attention_drug_b = SelfEncoder_D(input_size_drug, hidden_self_size, intermediate_size, num_heads, dropout, dropout)
        self.self_attention_cell = SelfEncoder_D(input_size_cell, hidden_self_size, intermediate_size, num_heads, dropout, dropout)
        
      
        # self.cross_attention_drug_a_cell = Attention_CD(hidden_self_size, hidden_cross_size, num_heads, dropout, dropout)
        # self.cross_attention_drug_b_cell = Attention_CD(hidden_self_size, hidden_cross_size, num_heads, dropout, dropout)
        
    
        self.pre = PreNN(hidden_cross_size * 3)
    def forward(self, drug_a, drug_b, cell):
      
        drug_a = drug_a.float()
        drug_b = drug_b.float()
        cell = cell.float()
        drug_a = drug_a.unsqueeze(1)  # (batch_size, 1, input_size_drug)
        drug_b = drug_b.unsqueeze(1)  # (batch_size, 1, input_size_drug)
        cell = cell.unsqueeze(1)      # (batch_size, 1, input_size_cell)

        drug_a, drug_a_mask = drug_feat(drug_a, device)
        drug_b, drug_b_mask = drug_feat(drug_b, device)
        cell, cell_mask = drug_feat(cell, device)
 

   
        drug_a_features, _ = self.self_attention_drug_a(drug_a, drug_a_mask)
        drug_b_features, _ = self.self_attention_drug_b(drug_b, drug_b_mask)
        cell_features, _ = self.self_attention_cell(cell, cell_mask)
        
    
     

        combined_features = torch.cat([drug_a_features,drug_b_features,cell_features], dim=-1)
        
      
        output = self.pre(combined_features)
        return output

In [ ]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import  Dataset


num_epochs = 100
batch_size = 32
learning_rate = 1e-4
final_scores = []

train_losses = []
val_losses = []

main_data = pd.read_csv("data/Merk/ONEIL_SCORE_Processed.csv")

drug_features_data = pd.read_csv("data/Graphlet_features_6.csv")

drug_features = drug_features_data.set_index('PubChem_CID').iloc[:, 5:]
# Load cell line features
cell_features_data = pd.read_csv("data/Merk/ONEIL_CELL_LINE_EXPRESSION.csv")
# Use 'gene_id' column as index for easy lookup
cell_features = cell_features_data.set_index('Cell_Line')

main_data = pd.read_csv("data/Merk/ONEIL_SCORE_Processed.csv")

drug_features_data = pd.read_csv("data/Graphlet_features_6.csv")

drug_features = drug_features_data.set_index('PubChem_CID').iloc[:, 5:]
# Load cell line features
cell_features_data = pd.read_csv("data/Merk/ONEIL_CELL_LINE_EXPRESSION.csv")

# Initialize lists to store features
drug_a_feature_list = []
drug_b_feature_list = []
cell_feature_list = []
labels = []

# Iterate through each sample in main data
for idx, row in main_data.iterrows():
    # Get drug1 and drug2 smiles
    drug1_smile = row['Drug_A']
    drug2_smile = row['Drug_B']
    cell_line = row['Cell_Line']
    
    # Lookup drug features
    try:
        drug_a_features = drug_features.loc[drug1_smile].values
        drug_b_features = drug_features.loc[drug2_smile].values
    except KeyError as e:
        print(f"Missing drug features for: {e}")
        continue
    
    # Lookup cell features
    try:
        cell_line_features = cell_features.loc[cell_line].values
    except KeyError as e:
        print(f"Missing cell line features for: {e}")
        continue
    
    # Store features and label
    drug_a_feature_list.append(drug_a_features)
    drug_b_feature_list.append(drug_b_features)
    cell_feature_list.append(cell_line_features)
    labels.append(row['Label'])

# Convert to tensors
drug_a_features = torch.tensor(np.array(drug_a_feature_list), dtype=torch.float32)
drug_b_features = torch.tensor(np.array(drug_b_feature_list), dtype=torch.float32)
cell_features = torch.tensor(np.array(cell_feature_list), dtype=torch.float32)
targets = torch.tensor(np.array(labels), dtype=torch.float32)


all_data = []


for i in range(len(targets)):
    all_data.append([
        drug_a_features[i].numpy(), 
        drug_b_features[i].numpy(),
        cell_features[i].numpy(),
        targets[i].item()           
    ])

print(f"数据已成功组合，共 {len(all_data)} 个样本。")


class DrugDataset(Dataset):
    def __init__(self, drug_a, drug_b, cell, target):
        self.drug_a = drug_a
        self.drug_b = drug_b
        self.cell = cell
        self.target = target
        
    def __len__(self):
        return len(self.target)
    
    def __getitem__(self, idx):
        return self.drug_a[idx], self.drug_b[idx], self.cell[idx], self.target[idx]


dataset = DrugDataset(drug_a_features, drug_b_features, cell_features, targets)



In [ ]:
from sklearn import metrics
import numpy as np
import torch
from sklearn.metrics import confusion_matrix,recall_score,matthews_corrcoef,roc_curve,auc,precision_recall_curve
from sklearn.metrics import cohen_kappa_score, accuracy_score, roc_auc_score, precision_score, balanced_accuracy_score


def calculateScore(y, pred_y):

    y = np.array(y).squeeze()
    pred_y = np.array(pred_y).squeeze()


    pred_y_binary = (pred_y > 0.5).astype(int)


    cm = confusion_matrix(y, pred_y_binary)

    if cm.shape == (1,1):
        if y[0] == 0:
            cm = np.array([[cm[0,0], 0], [0, 0]])
        else:
            cm = np.array([[0, 0], [0, cm[0,0]]])
    elif cm.shape == (2,1) or cm.shape == (1,2):
        cm = np.array([[0, 0], [0, 0]])  


    tn, fp, fn, tp = cm.ravel()


    try:
        ROCArea = roc_auc_score(y, pred_y)
    except ValueError:
        ROCArea = 0.5 


    fpr, tpr, thresholds = roc_curve(y, pred_y)


    pre, rec, _ = precision_recall_curve(y, pred_y)
    PR_AUC = metrics.auc(rec, pre)

 
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = sensitivity  # recall = sensitivity
    F1Score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    BACC = balanced_accuracy_score(y, pred_y_binary)
    KAPPA = cohen_kappa_score(y, pred_y_binary)

    return {
        'confusion_matrix': cm.tolist(),  
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),  
        'sn': sensitivity,
        'sp': specificity,
        'bacc': BACC,
        'acc': accuracy,
        'AUC': ROCArea,
        'precision': precision,
        'F1': F1Score,
        'AUC_prec_rec': PR_AUC,
        'kappa': KAPPA,
        'true_labels': y.tolist(),      
        'pred_probs': pred_y.tolist()   
    }

In [ ]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Subset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import seaborn as sns


train_losses = []
val_losses = []
val_accs = []         
final_scores = []


labels = dataset.target.numpy() if isinstance(dataset.target, torch.Tensor) else dataset.target


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
labels = dataset.target.numpy() if isinstance(dataset.target, torch.Tensor) else dataset.target


if not np.issubdtype(labels.dtype, np.integer):
    print(f"Warning: Converting labels from {labels.dtype} to int64")
    labels = labels.astype(np.int64)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\nFold {fold+1}/5")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}")
    print(f"Train class distribution: {np.bincount(labels[train_idx])}")
    print(f"Val class distribution: {np.bincount(labels[val_idx])}")


    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

  
    drug_a_train = torch.stack([train_subset[i][0] for i in range(len(train_subset))])
    drug_b_train = torch.stack([train_subset[i][1] for i in range(len(train_subset))])
    cell_train = torch.stack([train_subset[i][2] for i in range(len(train_subset))])

    scaler_drug_a = StandardScaler()
    scaler_drug_b = StandardScaler()
    scaler_cell = StandardScaler()

    drug_a_train_scaled = scaler_drug_a.fit_transform(drug_a_train.numpy())
    drug_b_train_scaled = scaler_drug_b.fit_transform(drug_b_train.numpy())
    cell_train_scaled = scaler_cell.fit_transform(cell_train.numpy())

    drug_a_val = scaler_drug_a.transform(dataset.drug_a[val_idx].numpy())
    drug_b_val = scaler_drug_b.transform(dataset.drug_b[val_idx].numpy())
    cell_val = scaler_cell.transform(dataset.cell[val_idx].numpy())

    train_dataset = DrugDataset(
        torch.tensor(drug_a_train_scaled, dtype=torch.float32),
        torch.tensor(drug_b_train_scaled, dtype=torch.float32),
        torch.tensor(cell_train_scaled, dtype=torch.float32),
        dataset.target[train_idx]
    )

    val_dataset = DrugDataset(
        torch.tensor(drug_a_val, dtype=torch.float32),
        torch.tensor(drug_b_val, dtype=torch.float32),
        torch.tensor(cell_val, dtype=torch.float32),
        dataset.target[val_idx]
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


    model = GraphCrossSynergy().to(device)
   
    pos_weight_value = 4.5
    pos_weight = torch.tensor([pos_weight_value]).to(device)  
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
    l2_lambda = 1e-5


    epoch_train_losses = []
    epoch_val_losses = []
    epoch_val_accs = []
    epoch_val_f1s = []

    best_val_loss = float('inf')
    best_f1 = 0
    early_stop_counter = 0
    early_stop_patience = 10
    min_delta = 1e-4

 
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0

        for drug_a, drug_b, cell, target in train_loader:
            drug_a = drug_a.to(device)
            drug_b = drug_b.to(device)
            cell = cell.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            outputs = model(drug_a, drug_b, cell).squeeze()
            loss = criterion(outputs, target)

 
            l2_reg = torch.tensor(0.0).to(device)
            for param in model.parameters():
                if param.requires_grad:
                    l2_reg += torch.norm(param, 2)
            loss += l2_lambda * l2_reg

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        epoch_train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0
        val_targets_epoch, val_outputs_epoch = [], []
        val_preds_epoch = []  
        with torch.no_grad():
            for drug_a, drug_b, cell, target in val_loader:
                drug_a = drug_a.to(device)
                drug_b = drug_b.to(device)
                cell = cell.to(device)
                target = target.to(device)

                outputs = model(drug_a, drug_b, cell).squeeze()
                val_loss += criterion(outputs, target).item()
                val_targets_epoch.extend(target.cpu().numpy())
                val_outputs_epoch.extend(outputs.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        epoch_val_losses.append(avg_val_loss)
       
        epoch_score = calculateScore(np.array(val_targets_epoch), np.array(val_outputs_epoch))
        epoch_val_accs.append(epoch_score['acc'])
        # epoch_val_accs.append(epoch_score['acc'])
        # epoch_val_f1s.append(epoch_score['F1'])

        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, ACC: {epoch_score['acc']:.4f}")
        # print(f"ACC: {epoch_score['acc']:.4f}, F1: {epoch_score['F1']:.4f}, Best Threshold: {best_threshold:.3f}")
        scheduler.step(avg_val_loss)

        # # Early stopping
        if abs(best_val_loss - avg_val_loss) < min_delta:
            early_stop_counter += 1
            print(f"No significant improvement in val loss for {early_stop_counter} epochs.")
            if early_stop_counter >= early_stop_patience:
                print("Early stopping triggered.")
                break
        else:
            best_val_loss = avg_val_loss
            early_stop_counter = 0


   
    train_losses.append(epoch_train_losses)
    val_losses.append(epoch_val_losses)
    val_accs.append(epoch_val_accs)

    
    val_targets, val_outputs = [], []
    with torch.no_grad():
        for drug_a, drug_b, cell, target in val_loader:
            drug_a = drug_a.to(device)
            drug_b = drug_b.to(device)
            cell = cell.to(device)
            outputs = model(drug_a, drug_b, cell).squeeze()
            val_targets.extend(target.cpu().numpy())
            val_outputs.extend(outputs.cpu().numpy())

    score = calculateScore(np.array(val_targets), np.array(val_outputs))
    final_scores.append(score)
    print(f"\nFold {fold+1} Final Score: {score}")


metrics = ['acc', 'AUC', 'AUC_prec_rec', 'F1', 'bacc', 'precision', 'sn', 'sp', 'kappa']
for metric in metrics:
    scores = [score[metric] for score in final_scores]
    print(f"Final Average {metric.upper()}: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

def plot_metrics(metrics, title):
    plt.figure(figsize=(10, 6))
    for i in range(5):
        plt.plot(range(len(metrics[i])), metrics[i], label=f'Fold {i+1}')
    plt.xlabel('Epochs')
    plt.ylabel(title.split()[-1])
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_metrics(train_losses, 'Training Loss Progress')
plot_metrics(val_losses, 'Validation Loss Progress')
plot_metrics(val_accs, 'Validation ACC Progress')